# MotherBabyInsight 项目复现 Notebook

本 Notebook 用于在 Windows 或 VMware Ubuntu 中复现实训项目：环境检查、数据检查、运行主流程、验证输出文件，并嵌入可视化大屏。

In [ ]:
from pathlib import Path
import sys, platform, subprocess, json

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print('项目目录:', ROOT)
print('Python:', sys.version)
print('系统:', platform.platform())

## 1. 原始数据检查

确认两个原始 CSV 文件位于 `data/raw/`，不要移动或删除原始数据。

In [ ]:
raw_files = [
    ROOT / 'data/raw/sam_tianchi_mum_baby.csv',
    ROOT / 'data/raw/sam_tianchi_mum_baby_trade_history.csv',
]
for file in raw_files:
    print(file.relative_to(ROOT), '存在' if file.exists() else '缺失')

## 2. 数据预处理与特征工程说明

主程序会完成日期转换、缺失值填充、画像左连接、商品大类中文映射、频次编码、时间特征提取和 high_buy 辅助标签构造。

In [ ]:
import pandas as pd
trade = pd.read_csv(ROOT / 'data/raw/sam_tianchi_mum_baby_trade_history.csv')
baby = pd.read_csv(ROOT / 'data/raw/sam_tianchi_mum_baby.csv')
print('原始交易记录数:', len(trade))
print('宝宝画像记录数:', len(baby))
display(trade.head(3))
display(baby.head(3))

## 3. 运行完整项目

该步骤会生成清洗数据、模型结果、9 张报告图、报告附录图表、报告表格、`dashboard_data.json` 和 `dashboard.html`。

In [ ]:
cmd = [sys.executable, str(ROOT / 'src/mother_baby_insight.py')]
print('运行命令:', ' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)

## 4. 输出文件检查

In [ ]:
outputs = [
    ROOT / 'outputs/dashboard_data.json',
    ROOT / 'outputs/html/dashboard.html',
    ROOT / 'outputs/charts/09_wordcloud.png',
    ROOT / 'outputs/report_figures/figure_index.csv',
    ROOT / 'outputs/report_tables/table_data_overview.csv',
]
for file in outputs:
    print(file.relative_to(ROOT), '已生成' if file.exists() else '未生成')

In [ ]:
data = json.loads((ROOT / 'outputs/dashboard_data.json').read_text(encoding='utf-8'))
data['metrics']

## 5. 嵌入可视化大屏

如果 Jupyter 对本地 HTML 的相对资源路径有限制，请使用下一节的 HTTP 服务方式打开。

In [ ]:
from IPython.display import IFrame, display
display(IFrame(src=str(ROOT / 'outputs/html/dashboard.html'), width='100%', height=820))

## 6. VMware / Linux 推荐打开方式

在终端中运行：

```bash
cd ~/mother_baby_analysis
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt
python src/mother_baby_insight.py
cd outputs
python3 -m http.server 8000
```

然后访问：`http://localhost:8000/html/dashboard.html`。

如需启用 AI 专家分析接口，另开终端运行：

```bash
export LLM_API_KEY="your_api_key_here"
export LLM_API_BASE="https://api.openai.com/v1"
export LLM_MODEL="gpt-4o-mini"
python app.py
```